# MAE pretraining

Пайплайн предобучения MAE-энкодера на термо-кадрах.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parents[1]
sys.path.insert(0, str(PROJECT_ROOT))

print("cwd:", PROJECT_ROOT)

In [ ]:
import torch

import pytorch_lightning as pl
from pytorch_lightning.callbacks import ModelCheckpoint
from pytorch_lightning.loggers import CSVLogger
from torch.utils.data import DataLoader

from datasets import TermoFrameDataset
from datasets.transforms import Compose, HorizontalFlip, VerticalFlip, RandomRotate90

from models.pretraining.lightning_module import MAELightningModule

## DataLoader'ы

`train_datasets`/`val_datasets` — разные поддатасеты/видео, чтобы val не совпадал с train.

In [ ]:
transform = Compose([
    HorizontalFlip(p=0.5),
    VerticalFlip(p=0.5),
    RandomRotate90(p=0.5),
])

train_ds = TermoFrameDataset(root_dir=str(PROJECT_ROOT / "datasets" / "datasets_list"), include="dataset_kaggle", transform=transform)
val_ds = TermoFrameDataset(root_dir=str(PROJECT_ROOT / "datasets" / "datasets_list"), include="dataset_tpu", transform=None)

train_loader = DataLoader(train_ds, batch_size=16, shuffle=True, num_workers=16, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=16, shuffle=False, num_workers=16, pin_memory=True)

## Модель

In [ ]:
model = MAELightningModule(
    img_size=256, patch_size=16, in_channels=1,
    embed_dim=128, depth=2, num_heads=4,
    decoder_dim=64, decoder_depth=1, decoder_heads=2,
    mask_ratio=0.4, lr=1.5e-4,
)

checkpoint_callback = ModelCheckpoint(
    monitor="val_loss",
    mode="min",
    save_last=True,
    filename="epoch{epoch:02d}-loss{val_loss:.4f}",
)
logger = CSVLogger("logs", name="mae_pretrain")

trainer = pl.Trainer(
    max_epochs=10, accelerator="cpu",
    devices=1, callbacks=[checkpoint_callback],
    logger=logger,
)

In [ ]:
trainer.fit(model, train_loader, val_loader)

## Перенос энкодера в downstream-задачу

In [ ]:
pretrained_encoder_state = model.model.encoder.state_dict()
torch.save(pretrained_encoder_state, "checkpoints/mae_encoder.pt")